# A-Maze-ing

## Read the configuration file

Just open the file, split the options turn them into a dictionary.

In [1]:
with open("config.txt") as f:
    txt = f.read()
    opts = txt.split("\n")
    if opts[-1] == "":
        opts.pop()
    tup = [tuple(i.split("=")) for i in opts]
    conf = {i[0].upper(): i[1] for i in tup if i[0][0] != "#"}
print(conf)
try:
    size = (int(conf["HEIGHT"]), int(conf["WIDTH"]))
    start = conf["ENTRY"].split(",")
    end = conf["EXIT"].split(",")
    entr = (int(start[1]), int(start[0]))
    exit = (int(end[1]), int(end[0]))
except ValueError:
    print("[ERROR] Configurtion is in wrong format. Please, use integers")
except IndexError:
    print("[ERROR] Entry or exit in wrong format. Usage ex.: EXIT=int,int")

{'HEIGHT': '31', 'WIDTH': '31', 'ENTRY': '0,0', 'EXIT': '30,30', 'OUTPUT_FILE': 'maze.txt', 'PERFECT': 'False', 'ALGORITHM': 'Wilson'}


## Create a $h \times w$ array

Two different arrays:

* The maze itself.
* The visited cells.

The 42 was hardcoded.

In [2]:
from numpy import zeros, ones


size = (int(conf["HEIGHT"]), int(conf["WIDTH"]))
maze = ones(size, int) * 15
visited = zeros(size, int)
perfect = conf["PERFECT"].lower().capitalize() != "False"

if visited.shape[0] < 6 or visited.shape[1] < 8:
    is_42 = 0
    print("The '42' cannot be printed in a maze this size.")
else:
    is_42 = 1
    hs, ws = (visited.shape[0] - 1) // 2 - 2, (visited.shape[1] - 1) // 2 - 3
    pos_42 = (
        (hs, ws), (hs + 1, ws), (hs + 2, ws), (hs + 2, ws + 1),
        (hs + 2, ws + 2), (hs + 3, ws + 2), (hs + 4, ws + 2),
        (hs, ws + 4), (hs, ws + 5), (hs, ws + 6), (hs + 1, ws + 6),
        (hs + 2, ws + 6), (hs + 2, ws + 5), (hs + 2, ws + 4),
        (hs + 3, ws + 4), (hs + 4, ws + 4), (hs + 4, ws + 5),
        (hs + 4, ws + 6)
    )
    for i in pos_42:
        visited[i[0]][i[1]] = -1
print(pos_42)
print(visited)
for i in maze:
    print(i)

((13, 12), (14, 12), (15, 12), (15, 13), (15, 14), (16, 14), (17, 14), (13, 16), (13, 17), (13, 18), (14, 18), (15, 18), (15, 17), (15, 16), (16, 16), (17, 16), (17, 17), (17, 18))
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 

## Manage errors

Checks:

* Maze dimensions must be positive.
* Maze entrance and exit are not in the same tile.
* Maze entrance and exit are inside the maze.
* Maze entrance and exit are not inside forbidden tiles.
* All maze walls in a tile are coherent with the neighbouring tiles.

In [3]:
from numpy.typing import NDArray


class MazeError(Exception):
    def __init__(self, msg: str):
        self.msg = msg


start = conf["ENTRY"].split(",")
end = conf["EXIT"].split(",")
entr = (int(start[1]), int(start[0]))
exit = (int(end[1]), int(end[0]))

if entr == exit:
    raise MazeError("Maze entrance and exit must be different")

if size[0] <= 0 or size[1] <= 0:
    raise MazeError("Maze dimensions must be positive")

in_bounds = [
    entr[0] >= 0,
    entr[1] >= 0,
    entr[0] < size[0],
    entr[1] < size[1],
    exit[0] >= 0,
    exit[1] >= 0,
    exit[0] < size[0],
    exit[1] < size[1]
]
if False in in_bounds:
    raise MazeError("Entrance or exit of the maze seem to be out of bounds")

if visited.shape[0] > 5 or visited.shape[1] > 7:
    if True in [entr in pos_42, exit in pos_42]:
        raise MazeError("Entrance or exit of the maze seem to be inside the 42")


def get_walls(n: int) -> tuple[int]:
    if n > 15 or n < 0:
        print(n)
        raise ValueError
    bits = []
    bits.append(n // 8)
    n %= 8
    bits.append(n // 4)
    n %= 4
    bits.append(n // 2)
    n %= 2
    bits.append(n)
    bits.reverse()
    return tuple(bits)


def check_tile(maze: NDArray, coords: tuple[int]):
    i, j = coords
    tile = int(maze[i, j])
    msg = f"Maze tile {coords} has incoherent walls"
    walls = get_walls(tile)
    if i == 0:
        if walls[0] == 0:
            raise MazeError(f"{msg} with north border")
    else:
        if walls[0] != get_walls(maze[i - 1][j])[2]:
            raise MazeError(f"{msg} with tile {(i - 1, j)}")
    if i == maze.shape[0] - 1:
        if walls[2] == 0:
            raise MazeError(f"{msg} with south border")
    else:
        if walls[2] != get_walls(maze[i + 1][j])[0]:
            raise MazeError(f"{msg} with tile {(i + 1, j)}")
    if j == 0:
        if walls[3] == 0:
            raise MazeError(f"{msg} with west border")
    else:
        if walls[3] != get_walls(maze[i][j - 1])[1]:
            raise MazeError(f"{msg} with tile {(i, j - 1)}")
    if j == maze.shape[1] - 1:
        if walls[1] == 0:
            raise MazeError(f"{msg} with east border")
    else:
        if walls[1] != get_walls(maze[i][j + 1])[3]:
            raise MazeError(f"{msg} with tile {(i, j + 1)}")

## Maze generator

Initial thought: Wilson algorithm.

* First tile in the maze.
* Random tile makes random walk.
* Looping random walk erases loop.
* Random walk finds maze: becomes part of the maze.
* Repeat until all tiles are part of the maze.

Second algorithm: Aldous-Broder:

* First tile in the maze.
* Point in there makes random walk.
* If unvisited cell: connects both tiles, continue random walk.
* Else: continue random walk.
* Repeat until all tiles are visited.

Complementary algorithm: DFS:

* First tile in the maze.
* Random walk until reaches a tile with no neighbours to visit.
* Bactrack to a tile with neighbours to visit, repeat.
* Repeat until all tiles are visited.

Complementary algorithm: Prim:

* First tile in the maze goes to a set.
* Select a yet to visit neighbour to a tile in the set.
* Connect both tiles.
* If tile has all neighbouring tiles visited, remove it from the set.
* Repeat until set is empty.

To do if I feel like it:

* Hunt & Kill (may be easy after DFS)
* Eller (doable)
* Krunskal (pain)
* Origin Shift (2 painful 4 me)

In [4]:
import random as rng


def manhattan(curr: tuple[int], neigh: tuple[int]) -> int:
    if curr == neigh:
        return 0
    return abs(curr[0] - neigh[0]) + abs(curr[1] - neigh[1])



def connect(maze: NDArray, curr: tuple[int], neigh: tuple[int]) -> None:
    if manhattan(curr, neigh) != 1:
        return
    if curr[0] == neigh[0] + 1 and get_walls(maze[curr[:]])[0] == 1:
        maze[curr[0]][curr[1]] -= 1
        maze[neigh[0]][neigh[1]] -= 4
    if curr[0] == neigh[0] - 1 and get_walls(maze[curr[:]])[2] == 1:
        maze[curr[0]][curr[1]] -= 4
        maze[neigh[0]][neigh[1]] -= 1
    if curr[1] == neigh[1] - 1 and get_walls(maze[curr[:]])[1] == 1:
        maze[curr[0]][curr[1]] -= 2
        maze[neigh[0]][neigh[1]] -= 8
    if curr[1] == neigh[1] + 1 and get_walls(maze[curr[:]])[3] == 1:
        maze[curr[0]][curr[1]] -= 8
        maze[neigh[0]][neigh[1]] -= 2


def direction() -> tuple[str, int]:
    if rng.random() < .5:
        axis = "V"
    else:
        axis = "H"
    if rng.random() < .5:
        step = -1
    else:
        step = 1
    return (axis, step)


def count_visited(visited: NDArray) -> int:
    n = 0
    for i in range(visited.shape[0]):
        for j in range(visited.shape[1]):
            if visited[i][j] == 1:
                n += 1
    return n


def wilson_erase(visited: NDArray, path: list[tuple], point: tuple[int]):
    while path[-1] != point:
        visited[path[-1][:]] = 0
        path.pop()


try:
    rng.seed(conf["SEED"])
except KeyError:
    pass
# rng.seed(1)

# Initialization
v = -1
while v == -1:
    current = (rng.randrange(0, size[0] - 1), rng.randrange(0, size[1] - 1))
    v = visited[current[:]]
visited[current[0]][current[1]] = 1

# Recursive backtracker
"""lst = [current]
while len(lst) > 0:
    nvis = set()
    h, w = current[:]
    neighs = ((h - 1, w), (h, w + 1), (h + 1, w), (h, w - 1))
    for i in neighs:
        try:
            if True in [i[j] in (-1, maze.shape[j]) for j in [0, 1]]:
                raise IndexError
            if visited[i[:]] == 0:
                nvis.add(i[:])
        except IndexError:
            continue
    if len(nvis) == 0:
        lst.pop()
    else:
        new = rng.choice([*nvis])
        nvis.clear()
        connect(maze, current, new)
        visited[new[:]] = 1
        lst.append(new)
    try:
        current = lst[-1]
    except IndexError:
        pass"""

# Prim Algorithm
"""opened = set()
opened.add(tuple(current))
while len(opened) > 0:
    nvis = set()
    h, w = current[:]
    neighs = ((h - 1, w), (h, w + 1), (h + 1, w), (h, w - 1))
    for i in neighs:
        try:
            if True in [i[j] in (-1, maze.shape[j]) for j in [0, 1]]:
                raise IndexError
            if visited[i[:]] == 0:
                nvis.add(i[:])
        except IndexError:
            continue
    if len(nvis) == 0:
        opened.remove(current)
    else:
        new = rng.choice([*nvis])
        nvis.clear()
        connect(maze, current, new)
        visited[new[:]] = 1
        opened.add(new)
    if len(opened) > 0:
        current = rng.choice([*opened])"""

# Alodus-Broder Algorithm
"""yet = set()
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        if visited[i, j] == 0:
            yet.add((i, j))
while len(yet) > 0:
    direct = direction()
    try:
        if direct[0] == "V":
            neighbour = (current[0] + direct[1], current[1])
            if current[0] + direct[1] in (-1, maze.shape[0]):
                raise IndexError
        else:
            neighbour = (current[0], current[1] + direct[1])
            if current[1] + direct[1] in (-1, maze.shape[1]):
                raise IndexError
    except IndexError:
        continue
    if visited[neighbour[0], neighbour[1]] == -1:
        continue
    elif visited[neighbour[0], neighbour[1]] == 0:
        connect(maze, tuple(current), tuple(neighbour))
        yet.discard(neighbour)
    current = neighbour
    visited[current[0], current[1]] = 1"""

# Wilson Algorithm
yet = set()
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        if visited[i, j] == 0:
            yet.add((i, j))
while len(yet) > 0:
    path = []
    current = rng.choices([*yet], k=1)[0]
    visited[current[0], current[1]] = 2
    path.append(current)
    last_direction = ()
    while current in yet:
        direct = direction()
        while last_direction == direct:
            direct = direction()
        try:
            if direct[0] == "V":
                if current[0] + direct[1] in (-1, maze.shape[0]):
                    raise IndexError
                neighbour = (current[0] + direct[1], current[1])
            else:
                if current[1] + direct[1]  in (-1, maze.shape[1]):
                    raise IndexError
                neighbour = (current[0], current[1] + direct[1])
        except IndexError:
            continue
        if visited[neighbour[0], neighbour[1]] == -1:
            continue
        elif visited[neighbour[0], neighbour[1]] == 1:
            path.append(neighbour)
            break
        elif neighbour in path:
            wilson_erase(visited, path, neighbour)
        current = neighbour
        visited[current[0], current[1]] = 2
        if current not in path:
            path.append(current)
    for tile in path:
        visited[tile[0], tile[1]] = 1
        yet.discard(tile)
    while len(path) > 1:
        connect(maze, path[-2], path[-1])
        current = path[-1]
        path.pop()
    path.clear()

base = "0123456789abcdef"
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        try:
            check_tile(maze, (i, j))
            print(base[maze[i][j]], end="")
        except MazeError as e:
            print(e.msg)
    print()
print(maze)
if 0 in visited:
    print("There are still unvisited tiles")

d13d1113d155553d51151117d39793b
96856eee9055578392ad2ec53c452c2
87abd1392c3b97eaaaabe95143d383a
87c2bec6abac2d142ec6d43ad0146ae
ad3aa97902ebc16bed3d556abac5787
c3c006d2ec7ad2947bc1557e843d147
9696a952917c3a83b83ab95787c7c3b
83ed403c2e93aaa86ea804392bd13c2
ec13beabc3eac2e857c2c7c68456c3e
d52c47e87ababed43d3c153d2bb93c7
d3c5517c14446b97a90503ad4286e97
d2bd507bed1116abc2ed2a83d44516b
9443bc3ab96ac784547d6aead17d292
87bc6b86aed2fbabfff956d2bc1786a
c387bc2b8552fc4057fad1386907ebe
bc2d012a8796fffefffe96ec56a9507
a943aeec2d6957fbfd55413b97803c3
aebaeb97ad3ad3fafffb92aa856eed2
c528528529447ad2d1106a86c793d3a
d146bc6d06bbd416bae87ec117e8382
bc3b857969443903c43c7bbac7d2eea
ad40017abeb946ec5529506ad3bc392
c556ae902d2c395511447a96d40786a
95556baec12baebd6abb9681790547a
85153a87bae84787946c696ebaad3ba
ed2d46c7ac547bc3eb913c3baeed282
b947917ba911505456aeed4443b906a
aa93ac3c46ae96915543d13b92c6c7a
c06869453b83c7ac397c7e82a851552
d2b87ab92ae8512d46d117ac2ad43d2
d6c6d46ec4547ec557d6c56d6c57c7e
[[13  1 

In [5]:
def break_wall(maze: NDArray, tile: tuple[int], op: str) -> None:
    i, j = tile[:]
    if op == "N":
        connect(maze, tile, (i - 1, j))
    elif op == "E":
        connect(maze, tile, (i, j + 1))
    elif op == "S":
        connect(maze, tile, (i + 1, j))
    elif op == "W":
        connect(maze, tile, (i, j - 1))


def remove_walls(maze: NDArray, tile: tuple[int], prob: float = 1) -> None:
    if maze[tile[:]] == 15 or prob < 0 or prob > 1:
        return
    i, j = tile[:]
    walls = get_walls(maze[tile[:]])
    r_walls = []
    if i > 0:
        if maze[i - 1][j] != 15 and walls[0] == 1:
            r_walls.append("N")
    if j < maze.shape[1] - 1:
        if maze[i][j + 1] != 15 and walls[1] == 1:
            r_walls.append("E")
    if i < maze.shape[0] - 1:
        if maze[i + 1][j] != 15 and walls[2] == 1:
            r_walls.append("S")
    if j > 0:
        if maze[i][j - 1] != 15 and walls[3] == 1:
            r_walls.append("W")
    if len(r_walls) > 0 and rng.random() < prob:
        break_wall(maze, tile, rng.choice(r_walls))


if perfect is False:
    print(maze)
    connect(maze, (0, 0), (0, 1))
    connect(maze, (0, 0), (1, 0))
    connect(maze, (0, maze.shape[1] - 1), (0, maze.shape[1] - 2))
    connect(maze, (0, maze.shape[1] - 1), (1, maze.shape[1] - 1))
    connect(maze, (maze.shape[0] - 1, maze.shape[1] - 1), (maze.shape[0] - 1, maze.shape[1] - 2))
    connect(maze, (maze.shape[0] - 1, maze.shape[1] - 1), (maze.shape[0] - 2, maze.shape[1] - 1))
    connect(maze, (maze.shape[0] - 1, 0), (maze.shape[0] - 1, 1))
    connect(maze, (maze.shape[0] - 1, 0), (maze.shape[0] - 2, 0))

    for _  in range(maze.shape[0] * maze.shape[1] // 3):
        tile = (rng.randrange(maze.shape[0]), rng.randrange(maze.shape[1]))
        remove_walls(maze, tile, .5)
    dead_ends = (7, 11, 13, 14)
    s_wall = ((0, 1), (0, 2), (0, 4), (0, 8))
    adm_walls = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]
    for i in range(maze.shape[0]):
        for j in range(maze.shape[1]):
            if maze[i, j] in dead_ends:
                remove_walls(maze, (i, j))
    for i in range(maze.shape[0]):
        for j in range(maze.shape[1]):
            if maze[i, j] == 0:
                adj_tiles = [(i - 1, j), (i, j + 1), (i + 1, j), (i, j - 1)]
                adj_walls = [maze[adj_tiles[k][:]] in s_wall[k] for k in range(len(adj_tiles))]
                if False not in adj_walls:
                    maze[i, j] = rng.choice(adm_walls)
                    walls = get_walls(maze[i, j])
                    if walls[0] == 1:
                        maze[i - 1][j] += 4
                    if walls[1] == 1:
                        maze[i][j + 1] += 8
                    if walls[2] == 1:
                        maze[i + 1][j] += 1
                    if walls[3] == 1:
                        maze[i][j - 1] += 2

for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        try:
            check_tile(maze, (i, j))
            print(base[maze[i][j]], end="")
        except MazeError as e:
            print(e.msg)
    print()

[[13  1  3 13  1  1  1  3 13  1  5  5  5  5  3 13  5  1  1  5  1  1  1  7
  13  3  9  7  9  3 11]
 [ 9  6  8  5  6 14 14 14  9  0  5  5  5  7  8  3  9  2 10 13  2 14 12  5
   3 12  4  5  2 12  2]
 [ 8  7 10 11 13  1  3  9  2 12  3 11  9  7 14 10 10 10 10 11 14  9  5  1
   4  3 13  3  8  3 10]
 [ 8  7 12  2 11 14 12  6 10 11 10 12  2 13  1  4  2 14 12  6 13  4  3 10
  13  0  1  4  6 10 14]
 [10 13  3 10 10  9  7  9  0  2 14 11 12  1  6 11 14 13  3 13  5  5  6 10
  11 10 12  5  7  8  7]
 [12  3 12  0  0  6 13  2 14 12  7 10 13  2  9  4  7 11 12  1  5  5  7 14
   8  4  3 13  1  4  7]
 [ 9  6  9  6 10  9  5  2  9  1  7 12  3 10  8  3 11  8  3 10 11  9  5  7
   8  7 12  7 12  3 11]
 [ 8  3 14 13  4  0  3 12  2 14  9  3 10 10 10  8  6 14 10  8  0  4  3  9
   2 11 13  1  3 12  2]
 [14 12  1  3 11 14 10 11 12  3 14 10 12  2 14  8  5  7 12  2 12  7 12  6
   8  4  5  6 12  3 14]
 [13  5  2 12  4  7 14  8  7 10 11 10 11 14 13  4  3 13  3 12  1  5  3 13
   2 11 11  9  3 12  7]
 [13  3 12  5  5  1 

## Maze solution

A\* algorithm:

```python
# G cost: distance from starting node
# H (heuristic) cost: distance from end node
# h_cost = abs(x_1 - x_2) + abs(y_1 - y_2)
# F cost: G cost + H cost
# open: set of nodes to be evaluated
# closed: set of already evaluated nodes

while True:
    current = "node in open with lowest f_cost"
    remove(current, open)
    add(current, closed)
    
    if current == exit:
        return
    
    for i in neighbours(current):
        if accessible(i) == 0 or i in closed:
            continue

        if f_cost(i) < f_cost(current) or i not in open:
            set_f_cost(i)
            current = parent(i)
            if i not in open:
                add(i, open)
```

In [6]:
from numpy import inf


class Node():
    def __init__(self, value):
        self.val = value
        self._g = inf
        self._h = inf
        self._f = inf
        self.next = None

    def add_front(self, newnode) -> None:
        if newnode is None:
            return
        newnode.next = self
        self = newnode

    def size(self) -> int:
        first = self
        n = 1
        while self.next != None and self.next is not first:
            n += 1
            self = self.next
        return n

    def get_g(self) -> int:
        return self._g

    def get_h(self) -> int:
        return self._h

    def get_f(self) -> int:
        return self._f

    def _set_f(self, f: int) -> None:
        self._f = f

    def set_g(self, g: int) -> None:
        self._g = g
        self._set_f(self.get_g() + self.get_h())

    def set_h(self, h: int) -> None:
        self._h = h
        self._set_f(self.get_g() + self.get_h())


def neighbours(maze: NDArray, current: tuple[int]) -> list[tuple]:
    n = maze[current[:]]
    neighs = []
    walls = get_walls(n)
    if walls[0] == 0:
        neighs.append((current[0] - 1, current[1]))
    if walls[1] == 0:
        neighs.append((current[0], current[1] + 1))
    if walls[2] == 0:
        neighs.append((current[0] + 1, current[1]))
    if walls[3] == 0:
        neighs.append((current[0], current[1] - 1))
    return neighs


def choose_current(f: NDArray, points: list[Node], exit: tuple[int]) -> Node:
    n = inf
    f_cost = []
    for i in points:
        p = i.val
        if p == exit:
            return i
        f_cost.append(f[p[:]])
    min_f = min(f_cost)
    to_select = []
    for j in range(len(f_cost)):
        if f_cost[j] == min_f:
            to_select.append(points[j])
    min_h = inf
    for k in to_select:
        h = manhattan(k.val, exit)
        if h < min_h:
            min_h = h
            current = k
    return current


def check_closed(closed: set[Node], point: tuple[int]) -> bool:
    for i in closed:
        if i.val == point:
            return True
    return False


def check_node(set: set[Node], point: tuple[int]) -> bool:
    for i in set:
        if i.val == point:
            return True
    return False


def rm_node(set: set[Node], point: tuple[int, int], new: Node) -> None:
    alt_path: list[Node] = []
    for n in set:
        if n.val == new.val:
            alt_path.append(n)

    for n in alt_path:
        if new.get_g() < n.get_g():
            set.remove(n)
    if len(alt_path) == 0:
        set.add(new)


"""opened, closed = set(), set()
f_cost = Inf * ones(maze.shape[:], int)
opened.add(Node(entr))
f_cost[entr[:]] = manhattan(entr, exit)
n = 0
while len(opened) > 0:
    n += 1
    current = choose_current(f_cost, list(opened), exit)
    opened.remove(current)
    closed.add(current)
    if current.val == exit:
        break
    for i in neighbours(maze, current.val):
        if check_closed(closed, i) is True:
            continue
        node = Node(i)
        node.next = current
        node.set_g(node.size() - 1)
        node.set_h(manhattan(i, exit))
        if node.get_f() < f_cost[i[:]]:
            f_cost[i[:]] = node.get_f()
            if check_node(opened, i) is True:
                if manhattan(i, entr) > node.get_g():
                    continue
            opened.add(node)"""
"""Initialize lists and F cost array"""
opened, closed = set(), set()
g_cost = inf * ones(maze.shape[:], int)
f_cost = inf * ones(maze.shape[:], int)
opened.add(Node(entr))
f_cost[entr[:]] = manhattan(entr, exit)
g_cost[entr[:]] = 0
n = 0
while len(opened) > 0:
    n += 1
    """Choose current node, move from open list to closed list"""
    current = choose_current(f_cost, list(opened), exit)
    opened.remove(current)
    closed.add(current)
    """If node is the exit, path has been found"""
    if current.val == exit:
        break
    for i in neighbours(maze, current.val):
        """If node is already closed, continue"""
        if check_node(closed, i) is True:
            continue
        node = Node(i)
        node.next = current
        node.set_g(node.size() - 1)
        if node.get_g() <  g_cost[i[:]]:
            g_cost[i[:]] = node.get_g()
            rm_node(opened, i, node)
        else:
            continue
        node.set_h(manhattan(i, exit))
        """Update F cost if is lower"""
        if node.get_f() < f_cost[i[:]] or check_node(opened, i) is False:
            f_cost[i[:]] = node.get_f()
            if check_node(opened, i) is False:
                opened.add(node)

"""Return the maze solution via backtracking"""
sol = ""
while current.next is not None:
    if current.val[0] == current.next.val[0] + 1:
        sol = "S" + sol
    elif current.val[0] == current.next.val[0] - 1:
        sol = "N" + sol
    elif current.val[1] == current.next.val[1] + 1:
        sol = "E" + sol
    elif current.val[1] == current.next.val[1] - 1:
        sol = "W" + sol
    current = current.next
print(n)
print(sol)

116
EESSSESSESSENEESESESSSEEESEEESSSSSSEESSSESSSSEEEESEEESSSSEEEEE


## Maze output

In [7]:
txt = ""
with open(conf['OUTPUT_FILE'], 'w') as out:
    print(maze)
    for i in range(size[0]):
        for j in range(size[1]):
            out.write(base[maze[i][j]])
        out.write("\n")

    out.write("\n")
    out.write(f"{entr[1]},{entr[0]}\n")
    out.write(f"{exit[1]},{exit[0]}\n")
    out.write(sol)

[[ 9  1  3  9  1  1  1  3  9  1  5  5  5  1  1  5  5  1  1  5  1  1  1  5
   5  3  9  3  9  1  3]
 [ 8  2  8  4  4  6 10 12  0  0  1  5  5  6  8  1  1  2 10  9  0  6 12  5
   3  8  0  4  2 12  2]
 [ 8  2  8  1  5  1  2  9  2  8  0  3  9  3 10 10 10 10 10 10 10  9  5  1
   4  2 12  1  0  3 10]
 [ 8  6 12  2  9  4  0  6 10 10 10 12  0  4  0  4  0  6 12  6 12  4  3 10
   9  0  1  4  6 10 10]
 [ 8  5  3 10 10  9  6  9  0  0  6  9  4  1  4  1  2  9  3  9  5  5  2 10
  10  8  0  5  5  0  2]
 [12  1  4  0  0  6  9  0  6  8  3 10  9  2  9  4  6 10 12  0  5  5  4  2
   8  4  2  9  1  0  6]
 [ 9  6  9  4  2  9  4  2  9  0  6 12  2 10  8  3  9  0  3 10  9  1  5  6
   8  3  8  6 12  0  3]
 [ 8  3 12  5  4  0  3 12  2 10  9  3 10  8  0  0  0  6 10  8  0  0  3  9
   0  0  4  1  3 12  2]
 [10 12  1  1  3 10 10  9  4  2 10 10 12  2  8  2  8  5  4  2 12  6 12  2
   8  4  1  6 12  3 10]
 [ 8  5  0  4  4  4  6  8  5  2 10 10  9  2 12  0  2  9  3 12  1  5  3 12
   0  3 10  9  3 12  6]
 [ 8  3  8  1  5  1 

In [8]:
with open("maze.txt") as f:
	line = f.readline()
	maze = []
	w = 0
	while line != "\n":
		maze.append(line.strip("\n"))
		if w == 0:
			w = len(line) - 1
		line = f.readline()
	en = f.readline().split(",")
	entry = (int(en[1]), int(en[0]))
	ex = f.readline().split(",")
	exit = (int(ex[1]), int(ex[0]))
	path = f.readline()

print(entry, exit)

(0, 0) (30, 30)


In [9]:
from maze_utils import get_walls


with open("maze.txt") as f:
	line = f.readline()
	maze = []
	w = 0
	while line != "\n":
		maze.append(line.strip("\n"))
		if w == 0:
			w = len(line) - 1
		line = f.readline()
	en = f.readline().split(",")
	entry = (int(en[1]), int(en[0]))
	ex = f.readline().split(",")
	exit = (int(ex[1]), int(ex[0]))
	path = f.readline()

print(entry, exit)

(0, 0) (30, 30)


In [10]:
pathtiles = []
curr = entry
for dir in path:
	if dir == "N":
		curr = (curr[0] - 1, curr[1])
	elif dir == "S":
		curr = (curr[0] + 1, curr[1])
	elif dir == "W":
		curr = (curr[0], curr[1] - 1)
	elif dir == "E":
		curr = (curr[0], curr[1] + 1)
	else:
		break
	if curr == exit:
		break
	pathtiles.append(curr)
pathtiles.append(exit)
pathtiles.insert(0, entry)

print(path)
print(pathtiles)

EESSSESSESSENEESESESSSEEESEEESSSSSSEESSSESSSSEEEESEEESSSSEEEEE
[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (3, 3), (4, 3), (5, 3), (5, 4), (6, 4), (7, 4), (7, 5), (6, 5), (6, 6), (6, 7), (7, 7), (7, 8), (8, 8), (8, 9), (9, 9), (10, 9), (11, 9), (11, 10), (11, 11), (11, 12), (12, 12), (12, 13), (12, 14), (12, 15), (13, 15), (14, 15), (15, 15), (16, 15), (17, 15), (18, 15), (18, 16), (18, 17), (19, 17), (20, 17), (21, 17), (21, 18), (22, 18), (23, 18), (24, 18), (25, 18), (25, 19), (25, 20), (25, 21), (25, 22), (26, 22), (26, 23), (26, 24), (26, 25), (27, 25), (28, 25), (29, 25), (30, 25), (30, 26), (30, 27), (30, 28), (30, 29), (30, 30)]


In [11]:
reset = "\x1b[0m"

black = "\x1b[40m"
red = "\x1b[41m"
green = "\x1b[42m"
yellow = "\x1b[43m"
blue = "\x1b[44m"
magenta = "\x1b[45m"
cyan = "\x1b[46m"
white = "\x1b[47m"

print(entry, exit)
for _ in range(2 * w + 1):
	print(f"{black}  {reset}", end="")
print()
h = len(maze)
i = 0
base = "0123456789abcdef"
for i in range(h):
	row = maze[i]
	print(f"{black}  {reset}", end="")
	for j in range(w):
		c = row[j]
		if c == "f":
			print(f"{red}  {reset}", end="")
		elif (i, j) == entry:
			print(f"{magenta}  {reset}", end="")
		elif (i, j) == exit:
			print(f"{green}  {reset}", end="")
		elif (i, j) in pathtiles:
			print(f"{blue}  {reset}", end="")
		else:
			print(f"{white}  {reset}", end="")
		if (i, j) in pathtiles and (i, j + 1) in pathtiles and get_walls(base.index(c))[1] == 0:
			print(f"{blue}  {reset}", end="")
		elif get_walls(base.index(c))[1] == 0:
			print(f"{white}  {reset}", end="")
		else:
			print(f"{black}  {reset}", end="")
	print()
	print(f"{black}  {reset}", end="")
	for j in range(w):
		c = row[j]
		if (i, j) in pathtiles and (i + 1, j) in pathtiles and get_walls(base.index(c))[2] == 0:
			print(f"{blue}  {reset}", end="")
		elif get_walls(base.index(c))[2] == 0:
			print(f"{white}  {reset}", end="")
		else:
			print(f"{black}  {reset}", end="")
		print(f"{black}  {reset}", end="")
	print()
print(path)

(0, 0) (30, 30)
                                                                                                                              
                                                                                                                              
                                                                                                                              
                                                                                                                              
                                                                                                                              
                                                                                                                              
                                                                                                                              
                                                                                               